In [1]:
# Defines function to parse the cleaned map to dict

import ast
import re

def parse_mappings(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    result = {}

    # Match each Columns declaration and its mapping block
    pattern = r"Columns:\s*(\[.*?\])\s*'[^']*':\s*\{(.*?)\}"

    for columns_str, mapping_str in re.findall(
        pattern, text, flags=re.DOTALL
    ):
        columns = ast.literal_eval(columns_str)

        # Parse key:value pairs
        mapping = {}
        for key, value in re.findall(
            r"([^,\s]+)\s*:\s*(\d+)\s*,?", mapping_str
        ):
            mapping[key] = int(value)

        for column in columns:
            result[column] = mapping

    return result

In [2]:
import pandas as pd
df_raw = pd.read_csv('train.csv')
df = df_raw.copy()

In [3]:
# Limpa colunas desnecessárias
colunas_remocao = [
    'Utilities',
    'Street',
    'Condition1',
    'Condition2',
    'RoofMatl',
    'Heating',
    'PoolQC',
    'Exterior1st',
    'Exterior2nd',
    'SaleType'
]
df = df.drop(columns=colunas_remocao)

In [4]:
# Mapeamento da coluna MSSubClass para descrições textuais
mssubclass_map = {
    20: '1-STORY 1946 & NEWER ALL STYLES',
    30: '1-STORY 1945 & OLDER',
    40: '1-STORY W/FINISHED ATTIC ALL AGES',
    45: '1-1/2 STORY - UNFINISHED ALL AGES',
    50: '1-1/2 STORY FINISHED ALL AGES',
    60: '2-STORY 1946 & NEWER',
    70: '2-STORY 1945 & OLDER',
    75: '2-1/2 STORY ALL AGES',
    80: 'SPLIT OR MULTI-LEVEL',
    85: 'SPLIT FOYER',
    90: 'DUPLEX - ALL STYLES AND AGES',
    120: '1-STORY PUD (Planned Unit Development) - 1946 & NEWER',
    150: '1-1/2 STORY PUD - ALL AGES',
    160: '2-STORY PUD - 1946 & NEWER',
    180: 'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER',
    190: '2 FAMILY CONVERSION - ALL STYLES AND AGES',
}
df['MSSubClass'] = df['MSSubClass'].map(mssubclass_map)

In [5]:
# Criação de coluna boolean para existência da garagem
df['GarageYrBlt_exists'] = df['GarageYrBlt'].notna().astype(int)

In [6]:
from sklearn.model_selection import train_test_split

# Separação das variáveis explicativas e variável alvo
X = df.drop(columns=['Id', 'SalePrice'])
y = df['SalePrice']

# Divide os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (1168, 70)
Teste: (292, 70)


In [7]:
# Colunas em que o nulo representa ausência do recurso
colunas_sem_recurso = [
    'Electrical',
    'Alley',
    'MasVnrType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageType',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'Fence',
    'MiscFeature'
]

# Substitui os nulos pela categoria "None"
for coluna in colunas_sem_recurso:
    if coluna in X_train.columns:
        X_train[coluna] = X_train[coluna].fillna('None')
        X_test[coluna] = X_test[coluna].fillna('None')

In [8]:
from sklearn.impute import SimpleImputer

# Identifica as colunas numéricas
num_columns = X_train.select_dtypes(include='number').columns

# Inicializa o imputador
imputer = SimpleImputer(strategy='mean')

# Ajusta no treino e transforma
X_train[num_columns] = imputer.fit_transform(X_train[num_columns])

# Aplica os mesmos valores de imputação no teste
X_test[num_columns] = imputer.transform(X_test[num_columns])

In [9]:
# - Transforme as colunas ordinais em números (Ex: {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}; )
mappings = parse_mappings("possible_values_map_clean.txt")
for column, mapping in mappings.items():
    if column in X_train.columns:
        X_train[column] = X_train[column].map(mapping)
        X_test[column] = X_test[column].map(mapping)

In [10]:
from sklearn.preprocessing import OneHotEncoder

# Identifica as colunas categóricas restantes
cat_columns = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns

# Inicializa o encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Ajusta no treino e transforma
X_train_cat = encoder.fit_transform(X_train[cat_columns])

# Aplica a mesma codificação no teste
X_test_cat = encoder.transform(X_test[cat_columns])

In [11]:
from sklearn.preprocessing import StandardScaler

# Remove as colunas categóricas originais
X_train_num = X_train.drop(columns=cat_columns)
X_test_num = X_test.drop(columns=cat_columns)

# Identifica as colunas numéricas que devem ser normalizadas
scale_columns = [
    col for col in X_train_num.columns
    if col != 'CentralAir'
    and col != 'GarageYrBlt_exists'
]

# Inicializa o normalizador
scaler = StandardScaler()

# Ajusta no treino e transforma
X_train_num_scaled = X_train_num.copy()
X_test_num_scaled = X_test_num.copy()

X_train_num_scaled[scale_columns] = scaler.fit_transform(
    X_train_num[scale_columns]
)

# Aplica os parâmetros do treino ao teste
X_test_num_scaled[scale_columns] = scaler.transform(
    X_test_num[scale_columns]
)

In [12]:
import numpy as np

# Combina os conjuntos
X_train_final = np.concatenate(
    [X_train_num_scaled.to_numpy(), X_train_cat], axis=1
)

X_test_final = np.concatenate(
    [X_test_num_scaled.to_numpy(), X_test_cat], axis=1
)

print("Treino:", X_train_final.shape)
print("Teste:", X_test_final.shape)

Treino: (1168, 169)
Teste: (292, 169)


In [13]:
# Transforma os preços com logaritmo natural
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Padroniza o alvo usando apenas os dados de treino
y_scaler = StandardScaler()

y_train_final = y_scaler.fit_transform(
    y_train_log.to_numpy().reshape(-1, 1)
).ravel()

y_test_final = y_scaler.transform(
    y_test_log.to_numpy().reshape(-1, 1)
).ravel()

print("Treino:", y_train_final.shape)
print("Teste:", y_test_final.shape)

Treino: (1168,)
Teste: (292,)


In [14]:
print("X_train:", X_train_final.shape)
print("X_test:", X_test_final.shape)

print("y_train:", y_train_final.shape)
print("y_test:", y_test_final.shape)

print("NaN em X_train:", np.isnan(X_train_final).sum())
print("NaN em X_test:", np.isnan(X_test_final).sum())

print("Inf em X_train:", np.isinf(X_train_final).sum())
print("Inf em X_test:", np.isinf(X_test_final).sum())

X_train: (1168, 169)
X_test: (292, 169)
y_train: (1168,)
y_test: (292,)
NaN em X_train: 0
NaN em X_test: 0
Inf em X_train: 0
Inf em X_test: 0


In [15]:
import torch

# Define o tipo de dado e converte para tensores
X_train_tensor = torch.tensor(
    X_train_final, dtype=torch.float32
)
X_test_tensor = torch.tensor(
    X_test_final, dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_final, dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test_final, dtype=torch.float32
).reshape(-1, 1)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([1168, 169])
y_train: torch.Size([1168, 1])
X_test: torch.Size([292, 169])
y_test: torch.Size([292, 1])


In [16]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

class ModeloCasasMLP(nn.Module):
    def __init__(
        self,
        input_size: int,
        n_camadas: int,
        neuronios: list[int],
        ativacoes: list[nn.Module],
        lr: float = 0.001,
        weight_decay: float = 0.0001,
        epochs: int = 100,
        patience: int = 20,
        conexoes_residuais: bool = False,
    ):
        super().__init__()

        # Validações
        if len(neuronios) != n_camadas:
            raise ValueError("A quantidade de elementos em neuronios deve ser igual a n_camadas.")

        if len(ativacoes) != n_camadas:
            raise ValueError("A quantidade de elementos em ativacoes deve ser igual a n_camadas.")
          
        if n_camadas <= 0:
            raise ValueError("O número de camadas deve ser maior que zero.")

        if not isinstance(neuronios, list) or not all(isinstance(n, int) for n in neuronios):
            raise ValueError("neuronios deve ser uma lista de inteiros.")

        if not isinstance(ativacoes, list) or not all(isinstance(a, nn.Module) for a in ativacoes):
            raise ValueError("ativacoes deve ser uma lista de instâncias de nn.Module.")
        
        self.n_camadas = n_camadas
        self.neuronios = neuronios
        self.ativacoes = ativacoes
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.patience = patience
        self.input_size = input_size
        self.conexoes_residuais = conexoes_residuais
        self.layers = nn.ModuleList()
        self.skip_connections = nn.ModuleList()
        
        # Construção das camadas ocultas
        tamanho_entrada = input_size
        for i in range(n_camadas):
            self.layers.append(
                nn.Linear(tamanho_entrada, neuronios[i])
            )

            self.layers.append(ativacoes[i])

            if conexoes_residuais:
                self.skip_connections.append(
                    nn.Identity()
                    if tamanho_entrada == neuronios[i]
                    else nn.Linear(tamanho_entrada, neuronios[i], bias=False)
                )

            tamanho_entrada = neuronios[i]

        # Camada de saída (regressão)
        self.output = nn.Linear(tamanho_entrada, 1)

        # Função de perda para regressão
        self.criterion = nn.MSELoss()
        
        # Otimizador
        self.optimizer = optim.Adam(
            self.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        
    def forward(self, x):
        for i in range(self.n_camadas):
            entrada = x
            x = self.layers[2 * i](x)
            x = self.layers[2 * i + 1](x)

            if self.conexoes_residuais:
                x = x + self.skip_connections[i](entrada)

        return self.output(x)
      
    def train_model(self, X_train, y_train, X_test, y_test, plot=False):
        train_losses = []
        test_losses = []
        
        best_test_loss = float("inf")
        epochs_no_improve = 0
        best_parameters_config = None
        
        for epoch in range(self.epochs):
            # Treinamento: atualiza os pesos
            self.train()

            self.optimizer.zero_grad()

            y_pred_train = self.forward(X_train)
            train_loss = self.criterion(y_pred_train, y_train)

            train_loss.backward()
            self.optimizer.step()

            train_losses.append(train_loss.item())

            # Avaliação: não atualiza os pesos
            self.eval()

            with torch.no_grad():
                y_pred_test = self.forward(X_test)
                test_loss = self.criterion(y_pred_test, y_test)

            test_losses.append(test_loss.item())

            if (epoch + 1) % 20 == 0:
                print(
                    f"Época [{epoch + 1}/{self.epochs}] "
                    f"| Treino: {train_loss.item():.6f} "
                    f"| Teste: {test_loss.item():.6f}"
                )
            # Early stopping
            if test_loss.item() < best_test_loss:
                best_test_loss = test_loss.item()
                epochs_no_improve = 0
                best_parameters_config = self.state_dict()
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= self.patience:
                print(f"Parando cedo na época {epoch + 1} devido à falta de melhoria no teste.")
                break

        if plot:
            plt.figure(figsize=(10, 5))
            plt.plot(train_losses, label="Treino")
            plt.plot(test_losses, label="Teste")
            plt.xlabel("Época")
            plt.ylabel("MSE")
            plt.title("Evolução da perda: treino vs. teste")
            plt.legend()
            plt.grid(True)
            plt.show()

        if best_parameters_config is not None:
            print("Carregando os melhores parâmetros encontrados durante o treinamento.")
            self.load_state_dict(best_parameters_config)
        return train_losses, test_losses
    
    def generate_metrics(self, X_test, y_test, print_results=True):
        # Gera previsões
        self.eval()

        with torch.no_grad():
            y_pred_scaled = self(X_test).cpu().numpy().reshape(-1)

        # Desfaz a padronização e o logaritmo do alvo
        y_pred_log = y_scaler.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()

        y_pred_price = np.expm1(y_pred_log)

        # Valores reais em reais
        if torch.is_tensor(y_test):
            y_true_scaled = y_test.detach().cpu().numpy().reshape(-1, 1)
            y_true_log = y_scaler.inverse_transform(y_true_scaled).ravel()
            y_true_price = np.expm1(y_true_log)
        else:
            y_true_price = y_test.to_numpy()

        # Métricas
        rmse = np.sqrt(mean_squared_error(y_true_price, y_pred_price))
        mae = mean_absolute_error(y_true_price, y_pred_price)
        r2 = r2_score(y_true_price, y_pred_price)

        if print_results:
            print(f"RMSE: R$ {rmse:,.2f}")
            print(f"MAE:  R$ {mae:,.2f}")
            print(f"R²:   {r2:.4f}")
        return rmse, mae, r2
    
    def print_model_info(self):
        print("Número de camadas:", self.n_camadas)
        print("Neuronios por camada:", self.neuronios)
        print("Ativacoes:", self.ativacoes)
        print("Epochs:", self.epochs)
        print("Learning rate:", self.lr)
        print("Weight decay:", self.weight_decay)
        print("Patience:", self.patience)
        print("Input size:", self.input_size)
        print("Conexões residuais:", self.conexoes_residuais)


In [26]:
hiperparametros_bateria_testes = {
  "n_testes": 500,
  "min_camadas": 1,
  "max_camadas": 5,
  "min_neuronios": 4,
  "max_neuronios": 64,
  "min_lr": 0.0001,
  "max_lr": 0.01,
  "min_weight_decay": 0.0001,
  "max_weight_decay": 0.01,
  "min_patience": 5,
  "max_patience": 50,
  "min_epochs": 50,
  "max_epochs": 500,
  "conexoes_residuais": [True, False]
}

In [27]:
# Activate GPU if available
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    X_train_tensor = X_train_tensor.to(device)
    y_train_tensor = y_train_tensor.to(device)
    X_test_tensor = X_test_tensor.to(device)
    y_test_tensor = y_test_tensor.to(device)



max_r2 = -np.inf

for i in range(hiperparametros_bateria_testes["n_testes"]):
    n_camadas = np.random.randint(hiperparametros_bateria_testes["min_camadas"], hiperparametros_bateria_testes["max_camadas"] + 1)
    neuronios = [np.random.randint(hiperparametros_bateria_testes["min_neuronios"], hiperparametros_bateria_testes["max_neuronios"] + 1) for _ in range(n_camadas)]
    lr = np.random.uniform(hiperparametros_bateria_testes["min_lr"], hiperparametros_bateria_testes["max_lr"])
    weight_decay = np.random.uniform(hiperparametros_bateria_testes["min_weight_decay"], hiperparametros_bateria_testes["max_weight_decay"])
    patience = np.random.randint(hiperparametros_bateria_testes["min_patience"], hiperparametros_bateria_testes["max_patience"] + 1)
    epochs = np.random.randint(hiperparametros_bateria_testes["min_epochs"], hiperparametros_bateria_testes["max_epochs"] + 1)
    conexoes_residuais = np.random.choice(hiperparametros_bateria_testes["conexoes_residuais"])

    model = ModeloCasasMLP(
        input_size=X_train_tensor.shape[1],
        n_camadas=n_camadas,
        neuronios=neuronios,
        ativacoes=[nn.ReLU() for _ in range(n_camadas)],
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        patience=patience,
        conexoes_residuais=conexoes_residuais
    )

    model.to(device)

    model.train_model(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, plot=False)
    rmse, mae, r2 = model.generate_metrics(X_test_tensor, y_test_tensor, print_results=False)
    if r2 > max_r2:
        max_r2 = r2
        best_model = model

print("Best model:")
print(best_model)
best_model.print_model_info()
rmse, mae, r2 = best_model.generate_metrics(X_test_tensor, y_test_tensor, print_results=True)

Época [20/232] | Treino: 0.145285 | Teste: 0.197758
Época [40/232] | Treino: 0.095270 | Teste: 0.144148
Época [60/232] | Treino: 0.077033 | Teste: 0.124715
Época [80/232] | Treino: 0.064289 | Teste: 0.115385
Época [100/232] | Treino: 0.055053 | Teste: 0.111884
Época [120/232] | Treino: 0.047738 | Teste: 0.109600
Época [140/232] | Treino: 0.041543 | Teste: 0.108568
Época [160/232] | Treino: 0.036345 | Teste: 0.109141
Parando cedo na época 161 devido à falta de melhoria no teste.
Carregando os melhores parâmetros encontrados durante o treinamento.
Época [20/401] | Treino: 0.115625 | Teste: 0.142975
Época [40/401] | Treino: 0.081269 | Teste: 0.121509
Época [60/401] | Treino: 0.062520 | Teste: 0.115640
Época [80/401] | Treino: 0.049065 | Teste: 0.111388
Época [100/401] | Treino: 0.040059 | Teste: 0.112881
Parando cedo na época 115 devido à falta de melhoria no teste.
Carregando os melhores parâmetros encontrados durante o treinamento.
Época [20/325] | Treino: 0.110773 | Teste: 0.150115
Épo